Public evidence copy. This file keeps the original Phase 4D seed-4 code and parameters. The version-controlled notebook had no embedded cell output, so this folder provides a separate verified summary and provenance note. Large checkpoints and posterior arrays are not included.


# Phase 4D seed 4: A100-authoritative full benchmark

This notebook preserves the completed Full GP and Direct GP results. It audits
the exact arrays those runs used, freezes one read-only authoritative NPZ, and
then runs all 13 DeepRV models with fresh NUTS chains on that NPZ. Phase 4A data
and Phase 4C posterior samples are excluded. The default is dry run only.


## 1. Validate the historical A100 environment


In [ ]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

import importlib
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=False)
required = ['jax', 'numpyro', 'flax', 'optax', 'orbax.checkpoint', 'arviz', 'scipy', 'matplotlib', 'dl4bi', 'dl4bi_sps']
missing = []
for package in required:
    try:
        importlib.import_module(package)
    except Exception as exc:
        missing.append((package, repr(exc)))

if missing:
    print('Installing because imports failed:')
    for package, exc in missing:
        print(' ', package, exc)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'dl4bi[benchmarks,cuda12] @ git+https://github.com/paytonni/dl4bi.git',
    ])
    raise SystemExit('Restart the Colab runtime, then rerun from this cell.')

import jax
import numpyro
device_kinds = [str(getattr(device, 'device_kind', device)) for device in jax.devices()]
print('jax:', jax.__version__)
print('numpyro:', numpyro.__version__)
print('backend:', jax.default_backend())
print('devices:', device_kinds)
print('XLA_PYTHON_CLIENT_PREALLOCATE:', os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'])
if jax.default_backend() != 'gpu' or not any('A100' in name.upper() for name in device_kinds):
    raise RuntimeError('Formal Phase 4D requires an NVIDIA A100 runtime.')


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Upload and validate the source ZIP


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
from zipfile import ZipFile

EXPECTED_SOURCE_SHA256 = '0814f62bc1b49bdc627e4abb179e4d6a17cd0992b3fa8d9b2621420da12d22a1'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Upload exactly phase4d_from_seed0_benchmark_sources.zip')
source_zip = Path('/content') / next(iter(uploaded))
if sha256_file(source_zip) != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('Source ZIP SHA-256 mismatch')
with ZipFile(source_zip) as handle:
    bad = handle.testzip()
    if bad is not None:
        raise RuntimeError(f'Source ZIP CRC failure: {bad}')
    handle.extractall('/content')
required = [
    '/content/phase4d_a100_authoritative_benchmark.py',
    '/content/paperlike_32x32_poisson_gp_pilot.py',
    '/content/experiments/phase4_64x64_three_seed_workflow.py',
    '/content/experiments/phase3d_64x64_failed_models_300k_mcmc.py',
]
missing = [name for name in required if not Path(name).is_file()]
if missing:
    raise RuntimeError(f'Missing extracted sources: {missing}')
print('Source package validation: PASS')


## 4. Configure exactly one seed and choose stages


In [ ]:
from pathlib import Path
import json

SELECTED_SEED = 4
DECODER_SEED = 0
DATA_SEED = SELECTED_SEED
NUTS_SEED = SELECTED_SEED

RUN_DRY_RUN = True
RUN_AUDIT_AND_FREEZE_DATA = False
RUN_DEEPRV_NUTS = False
RUN_DEEPRV_POSTERIOR_PREDICTION = False
RUN_POSTPROCESS = False
RUN_PACKAGE_RESULTS = False

# Permanent safety switches. Do not change them.
RUN_FULL_GP = False
RUN_DIRECT_GP = False
REUSE_PHASE4C_POSTERIOR = False

HISTORICAL_SEED0_OUTPUT_ROOT = Path(
    '/content/drive/MyDrive/dl4bi_colab_outputs/'
    'oliver_64x64_uniform_obs50_allmodels/outputs'
)
CHECKPOINT_ROOT = (
    HISTORICAL_SEED0_OUTPUT_ROOT
    / 'deeprv/_decoder_cache/'
    'target64_decoderseed0_inducing8-16-32_'
    'domainslowres_weightingsbilinear-cubic-dtc-fitc_ls30_steps200000'
)
OUTPUT_ROOT = Path(
    f'/content/drive/MyDrive/dl4bi_phase4/phase4d_final_benchmarks/seed_{SELECTED_SEED}'
)
LOCAL_WORK_ROOT = Path(f'/content/phase4d_seed_{SELECTED_SEED}_a100_authoritative')
RUNNER_PATH = Path('/content/phase4d_a100_authoritative_benchmark.py')
BASE_NOTEBOOK_PATH = Path('/content/paperlike_64x64_allmodels_optimized_colab.ipynb')
BASELINE_MANIFEST_PATH = Path('/content/phase4d_seed0_baseline_manifest.json')
CONFIG_PATH = LOCAL_WORK_ROOT / 'phase4d_a100_config.json'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)

assert SELECTED_SEED in (4, 12)
assert DECODER_SEED == 0 and DATA_SEED == NUTS_SEED == SELECTED_SEED
assert RUN_FULL_GP is False and RUN_DIRECT_GP is False
assert REUSE_PHASE4C_POSTERIOR is False
later_stages = [
    RUN_AUDIT_AND_FREEZE_DATA,
    RUN_DEEPRV_NUTS,
    RUN_DEEPRV_POSTERIOR_PREDICTION,
    RUN_POSTPROCESS,
    RUN_PACKAGE_RESULTS,
]
if RUN_DRY_RUN and any(later_stages):
    raise RuntimeError('Run dry run alone first. Then set RUN_DRY_RUN=False and enable later stages.')

config = {
    'selected_seed': SELECTED_SEED,
    'checkpoint_root': str(CHECKPOINT_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'local_work_root': str(LOCAL_WORK_ROOT),
    'base_notebook_path': str(BASE_NOTEBOOK_PATH),
    'baseline_manifest_path': str(BASELINE_MANIFEST_PATH),
    'prediction_draws_per_chain': 2000,
    'run_full_gp': False,
    'run_direct_gp': False,
    'reuse_phase4c_posterior': False,
}
CONFIG_PATH.write_text(json.dumps(config, indent=2, sort_keys=True) + '\n')
print('Selected seed:', SELECTED_SEED)
print('Immutable Full/Direct root:', OUTPUT_ROOT / 'direct_gp')
print('Authoritative NPZ:', OUTPUT_ROOT / f'seed_{SELECTED_SEED}_a100_authoritative_data.npz')
print('Fresh DeepRV root:', OUTPUT_ROOT / 'deeprv_a100_authoritative')


## 5. Historical subprocess helper


In [ ]:
import select
import subprocess
import sys

os.environ['DEEPRV_CHECKPOINTS_TO_KEEP'] = '1'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ.setdefault('DL4BI_LOCAL_LOG_DIR', str(LOCAL_WORK_ROOT / 'logs'))

def run_cmd(action, label):
    cmd = [sys.executable, '-u', str(RUNNER_PATH), action, '--config', str(CONFIG_PATH)]
    print('\n' + '=' * 100, flush=True)
    print('START:', label, flush=True)
    print('COMMAND:', ' '.join(cmd), flush=True)
    print('=' * 100, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env, bufsize=0)
    tail = ''
    fd = process.stdout.fileno()
    while True:
        ready, _, _ = select.select([fd], [], [], 0.5)
        if not ready:
            if process.poll() is not None:
                remainder = process.stdout.read() or b''
                if remainder:
                    text = remainder.decode('utf-8', errors='replace')
                    print(text, end='', flush=True)
                    tail = (tail + text)[-16000:]
                break
            continue
        chunk = os.read(fd, 8192)
        if not chunk:
            if process.poll() is not None:
                break
            continue
        text = chunk.decode('utf-8', errors='replace')
        print(text, end='', flush=True)
        tail = (tail + text)[-16000:]
    exit_code = process.wait()
    print('\nFINISHED:', label, 'exit_code=', exit_code, flush=True)
    if exit_code != 0:
        print('\nLAST CHILD OUTPUT:\n' + tail, flush=True)
        raise RuntimeError(f'{label} failed with exit code {exit_code}')


## 6. Stage 0: audit-only dry run (no freeze, no MCMC)


In [ ]:
if RUN_DRY_RUN:
    run_cmd('dry-run', f'A100-authoritative dry run, seed={SELECTED_SEED}')
else:
    print('Stage 0 disabled')


## 7. Stage 1: freeze internally identical A100 data


In [ ]:
if RUN_AUDIT_AND_FREEZE_DATA:
    run_cmd('audit-and-freeze', f'Audit and freeze A100 data, seed={SELECTED_SEED}')
else:
    print('Stage 1 disabled')


## 8. Stage 2: fresh NUTS for all 13 DeepRV models


In [ ]:
if RUN_DEEPRV_NUTS:
    run_cmd('run-deeprv', f'Fresh DeepRV NUTS 13/13, seed={SELECTED_SEED}')
else:
    print('Stage 2 disabled')


## 9. Stage 3: DeepRV posterior prediction


In [ ]:
if RUN_DEEPRV_POSTERIOR_PREDICTION:
    run_cmd('predict-deeprv', f'DeepRV posterior prediction, seed={SELECTED_SEED}')
else:
    print('Stage 3 disabled')


## 10. Stage 4: final metrics and complete ZIP


In [ ]:
if RUN_POSTPROCESS:
    run_cmd('postprocess', f'Final postprocessing, seed={SELECTED_SEED}')
else:
    print('Postprocessing disabled')

if RUN_PACKAGE_RESULTS:
    run_cmd('package', f'Complete result ZIP, seed={SELECTED_SEED}')
else:
    print('Packaging disabled')


## 11. Resume/status report


In [ ]:
run_cmd('status', f'Phase 4D status, seed={SELECTED_SEED}')
